# 09 — Trayectorias de distancia a la línea de pobreza (D33/D34)

Capa 2 del rediseño del eje del TFI: por cada par de hogares vinculado por el
panel (`data/03_panel/pares_h1`/`pares_h4`, D9), la variación de la distancia
a la línea de pobreza entre `t` y `t+h` (`src/trayectoria_distancia.py`) --
variación porcentual simétrica (objetivo principal, admite `ITF=0`), variación
del log (solo con `ITF>0` en ambas puntas) y descomposición ingreso/precios/
composición, componente común (mediana ponderada por trimestre y por
aglomerado-trimestre) y propia, tendencia previa (`t-1`->`t`, candidata de
feature) y los derivados `direccion`/`transicion`.

**Lee** `data/02b_distancia_lp/hogar/` (Capa 1, D34) y
`data/03_panel/{pares_h1,pares_h4}` (ya existentes, sin reconstruir). **Escribe**
`data/05_trayectoria_distancia/{pares_h1,pares_h4}/`. Solo cubre
`era="regional_2016"` (2016T2-2025T4) -- la isla histórica 2003-2006 (D35) se
construye aparte en `08b_retropolacion_pobreza.ipynb`, en sus propias
subcarpetas (`pares_h1_historico`/`pares_h4_historico`).

**No toca** `src/panel.py` ni `src/trayectorias.py` (D12/D13, intocables esta
ronda) -- lee sus salidas tal cual.

In [ ]:
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd

import pipeline_meta as pm
import panel
import trayectoria_distancia
from armonizacion import trimestres_publicados

DISTANCIA_LP = REPO / "data" / "02b_distancia_lp" / "hogar"
PANEL_DIR = REPO / "data" / "03_panel"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
META = REPO / "data" / "meta"
COMMIT_GIT = pm.obtener_commit_git(REPO)

TRIMESTRES_DISTANCIA = [t for t in trimestres_publicados() if t >= (2016, 2)]
print(f"{len(TRIMESTRES_DISTANCIA)} trimestres de origen candidatos (2016T2+)")


## Carga de la Capa 1 en memoria

Una sola lectura por trimestre -- se reusa tanto como origen `t` de un
horizonte como destino `t+h` de otro, sin releer del disco.

In [ ]:
distancia_por_trimestre = {}
for anio, trimestre in TRIMESTRES_DISTANCIA:
    p = DISTANCIA_LP / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    if p.exists():
        distancia_por_trimestre[(anio, trimestre)] = pd.read_parquet(p)
print(f"{len(distancia_por_trimestre)} particiones de distancia cargadas")


## Helper de enriquecimiento por par de origen

Mismo cálculo para `h=1` y `h=4`: pares + variaciones + descomposición
(`trayectoria_distancia.construir_pares_distancia`, incluye `par_valido`),
componente común/propio sobre `variacion_simetrica` y `variacion_log` por
trimestre de origen y por aglomerado-trimestre de origen, y tendencia previa
-- última observación anterior disponible del mismo hogar, no solo `t-1`
(D34 ampliada): un hogar en su 2ª visita en `t` tiene observación en `t-1`
(`gap_previo=0`); en su 3ª (tras el hueco de 2 trimestres) la tiene en `t-4`
(`gap_previo=3`, el tramo 1ª->2ª visita); en su 1ª, ninguna (`NA`).

In [ ]:
def construir_pares_enriquecidos(distancia_t: pd.DataFrame, distancia_th: pd.DataFrame,
                                   pares_hogar: pd.DataFrame, previo_1: pd.DataFrame | None,
                                   previo_4: pd.DataFrame | None) -> pd.DataFrame:
    pares = trayectoria_distancia.construir_pares_distancia(distancia_t, distancia_th, pares_hogar)
    for columna_variacion, sufijo in (("variacion_simetrica", "simetrica"), ("variacion_log", "log")):
        pares = trayectoria_distancia.agregar_componente_comun_y_propio(
            pares, columna_variacion, "PONDIH_t", ["ANO4_t", "TRIMESTRE_t"], f"trimestre_{sufijo}")
        pares = trayectoria_distancia.agregar_componente_comun_y_propio(
            pares, columna_variacion, "PONDIH_t", ["ANO4_t", "TRIMESTRE_t", "AGLOMERADO_t"], f"aglomerado_{sufijo}")
    return trayectoria_distancia.agregar_tendencia_previa(pares, previo_1, previo_4)


## `pares_h1` -- construcción autorreferencial

La tendencia previa de `t` necesita el propio `pares_h1` de `t-1`/`t-4`, ya
construido en una iteración anterior del mismo loop (los orígenes se procesan
en orden cronológico).

In [ ]:
t0 = time.time()
pares_h1_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
for anio, trimestre in TRIMESTRES_DISTANCIA:
    anio_th, trimestre_th = panel.sumar_trimestres(anio, trimestre, 1)
    if (anio, trimestre) not in distancia_por_trimestre or (anio_th, trimestre_th) not in distancia_por_trimestre:
        continue
    p_panel = pm.ruta_particion(PANEL_DIR, "pares_h1", anio, trimestre)
    if not p_panel.exists():
        continue
    pares_hogar = pd.read_parquet(p_panel)
    previo_1 = pares_h1_por_trimestre.get(panel.sumar_trimestres(anio, trimestre, -1))
    previo_4 = pares_h1_por_trimestre.get(panel.sumar_trimestres(anio, trimestre, -4))
    pares = construir_pares_enriquecidos(
        distancia_por_trimestre[(anio, trimestre)], distancia_por_trimestre[(anio_th, trimestre_th)],
        pares_hogar, previo_1, previo_4)
    pares_h1_por_trimestre[(anio, trimestre)] = pares
    destino = pm.ruta_particion(TRAYECTORIA, "pares_h1", anio, trimestre)
    destino.parent.mkdir(parents=True, exist_ok=True)
    pares.to_parquet(destino, index=False)

n_pares_h1 = sum(len(df) for df in pares_h1_por_trimestre.values())
duracion_h1 = time.time() - t0
print(f"pares_h1: {len(pares_h1_por_trimestre)} trimestres de origen, {n_pares_h1:,} pares, {duracion_h1:.1f}s")


## `pares_h4` -- tendencia previa contra el `pares_h1` ya completo

`h=4` reusa el diccionario `pares_h1_por_trimestre` ya terminado para la
tendencia previa (el tramo anterior es siempre de `pares_h1`, `t-1` o `t-4`,
independiente del horizonte del par principal). Un hogar que origina un par
`h=4` puede estar en su 1ª visita (`t=t0`, sin observación previa) o en su
2ª (`t=t0+1`, con observación en `t-1=t0`, `gap_previo=0`) -- nunca en su 3ª
(un origen `h=4` en la 3ª visita no tendría destino, D34).

In [ ]:
t0 = time.time()
pares_h4_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
for anio, trimestre in TRIMESTRES_DISTANCIA:
    anio_th, trimestre_th = panel.sumar_trimestres(anio, trimestre, 4)
    if (anio, trimestre) not in distancia_por_trimestre or (anio_th, trimestre_th) not in distancia_por_trimestre:
        continue
    p_panel = pm.ruta_particion(PANEL_DIR, "pares_h4", anio, trimestre)
    if not p_panel.exists():
        continue
    pares_hogar = pd.read_parquet(p_panel)
    previo_1 = pares_h1_por_trimestre.get(panel.sumar_trimestres(anio, trimestre, -1))
    previo_4 = pares_h1_por_trimestre.get(panel.sumar_trimestres(anio, trimestre, -4))
    pares = construir_pares_enriquecidos(
        distancia_por_trimestre[(anio, trimestre)], distancia_por_trimestre[(anio_th, trimestre_th)],
        pares_hogar, previo_1, previo_4)
    pares_h4_por_trimestre[(anio, trimestre)] = pares
    destino = pm.ruta_particion(TRAYECTORIA, "pares_h4", anio, trimestre)
    destino.parent.mkdir(parents=True, exist_ok=True)
    pares.to_parquet(destino, index=False)

n_pares_h4 = sum(len(df) for df in pares_h4_por_trimestre.values())
duracion_h4 = time.time() - t0
print(f"pares_h4: {len(pares_h4_por_trimestre)} trimestres de origen, {n_pares_h4:,} pares, {duracion_h4:.1f}s")

pm.registrar_corrida(META / "corridas.csv", "09_trayectorias_distancia", {"era": "regional_2016"},
                      len(pares_h1_por_trimestre) + len(pares_h4_por_trimestre), n_pares_h1 + n_pares_h4,
                      duracion_h1 + duracion_h4, COMMIT_GIT)


## Reporte: tendencia previa, descomposición, dirección y transición

Proporción de pares con `variacion_simetrica_previa` disponible (feature
candidata) y con descomposición definida (`ITF>0` en ambas puntas), por
horizonte -- y la distribución de `direccion`/`transicion`, solo para
dimensionar (el análisis descriptivo real va en `09b`).

In [ ]:
filas_reporte = []
for h, pares_por_trimestre in ((1, pares_h1_por_trimestre), (4, pares_h4_por_trimestre)):
    todos = pd.concat(pares_por_trimestre.values(), ignore_index=True)
    filas_reporte.append({
        "h": h, "n_pares": len(todos), "n_par_valido": int(todos["par_valido"].sum()),
        "pct_con_tendencia_previa": 100 * todos["variacion_simetrica_previa"].notna().mean(),
        "pct_gap_previo_0": 100 * (todos["gap_previo"] == 0).mean(),
        "pct_gap_previo_3": 100 * (todos["gap_previo"] == 3).mean(),
        "pct_con_descomposicion": 100 * todos["aporte_ingreso"].notna().mean(),
        "pct_direccion_mejora": 100 * (todos["direccion"] == "mejora").mean(),
        "pct_direccion_estable": 100 * (todos["direccion"] == "estable").mean(),
        "pct_direccion_empeora": 100 * (todos["direccion"] == "empeora").mean(),
    })
reporte_capa2 = pd.DataFrame(filas_reporte)
reporte_capa2.to_csv(META / "trayectoria_distancia_reporte_capa2.csv", index=False)
print("data/meta/trayectoria_distancia_reporte_capa2.csv escrito")
display(reporte_capa2.round(2))
